In [1]:
# ============
# Common
# ============

import sys
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd()).parent
# Connect to custom-defined modules
sys.path.append(str(PROJECT_ROOT))

%reload_ext autoreload
%autoreload 2

In [5]:
# ========================
# Test PPE
# ========================
from airport_ai.decision.ppe.selector import PersonSelector
from airport_ai.tracking.structures import TrackedObject
from airport_ai.decision.ppe.association import PPEAssociation

selector = PersonSelector()

association = PPEAssociation()

tracked_objects = [
    TrackedObject(
        track_id=1,
        class_id=0,
        class_name="person",
        confidence=0.98,
        x1=10,
        y1=20,
        x2=60,
        y2=180,
        center_x=35,
        center_y=100,
        width=50,
        height=160
    ),
    TrackedObject(
        track_id=2,
        class_id=4,
        class_name="airplane",
        confidence=0.99,
        x1=300,
        y1=100,
        x2=900,
        y2=500,
        center_x=600,
        center_y=300,
        width=600,
        height=400
    )
]

persons = selector.select(tracked_objects)

for person in persons:
    print(person.track_id, person.class_name)

statuses = association.associate(persons, tracked_objects)
for status in statuses:
    print(status)

1 person
PPEStatus(person=TrackedObject(track_id=1, class_id=0, class_name='person', confidence=0.98, x1=10, y1=20, x2=60, y2=180, center_x=35, center_y=100, width=50, height=160), safety_vest=False, ear_protection=False)


In [10]:
# ============================
# Test PPE Evaluator
# ============================
from airport_ai.decision.ppe.evaluator import PPEEvaluator
from airport_ai.decision.ppe.structures import PPEStatus

worker = TrackedObject(
    track_id=1,
    class_id=0,
    class_name="person",
    confidence=0.98,
    x1=100,
    y1=100,
    x2=200,
    y2=300,
    center_x=150,
    center_y=200,
    width=100,
    height=200
)

status = PPEStatus(
    person=worker,
    safety_vest=False,
    ear_protection=True
)

evaluator = PPEEvaluator(
    require_safety_vest=True,
    require_ear_protection=True
)

events = evaluator.evaluate([status])

for event in events:
    print(event)

PPEEvent(timestamp=datetime.datetime(2026, 7, 27, 19, 57, 43, 223000), track_id=1, object_type='person', event_type='Safety Vest Missing', severity='HIGH', message='Worker 1 is not wearing a safety vest.')


In [13]:
from airport_ai.decision.ppe.visualization import PPEVisualizer
visualizer = PPEVisualizer()


In [ ]:
# ===========================
# Complete PPE Pipeline
# ===========================
import cv2
from airport_ai.config.settings import *

from airport_ai.streams.camera import AsyncCamera

from airport_ai.tracking.tracker import YOLOTracker
from airport_ai.tracking.parser import TrackingParser

from airport_ai.decision.ppe.visualization import PPEVisualizer

camera = AsyncCamera(
    source=VIDEO_SOURCE,
    width=FRAME_WIDTH,
    height=FRAME_HEIGHT,
    queue_size=FRAME_QUEUE_SIZE
).start()

tracker = YOLOTracker(str(PROJECT_ROOT/"models/yolov8n.pt"))
parser = TrackingParser()
visualizer = PPEVisualizer()

while True:
    frame = camera.read()
    result = tracker.track(frame)
    tracked_objects = parser.parse(result)
    persons = selector.select(tracked_objects)
    statuses = association.associate(person, tracked_objects)
    events = evaluator.evaluate(statuses)
    frame = visualizer.draw(frame, statuses, events)
    cv2.imshow("PPE Compliance", frame)
    if cv2.waitKey(1) == ord("q"):
        break